In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit

PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "data" / "dataset_ml.csv").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATASET_PATH = PROJECT_DIR / "data" / "dataset_ml.csv"

df = pd.read_csv(DATASET_PATH, low_memory=False)

print("Dimensions :", df.shape)
print("Accidents uniques :", df["Num_Acc"].nunique())

Dimensions : (657408, 17)
Accidents uniques : 273226


In [7]:
features = [
    "sexe", "catu", "catv", "atm", "lum",
    "col", "choc", "manv", "plan", "surf",
    "nbv", "secu1", "infra", "age", "place"
]

target = "grav"

X = df[features].copy()
y = df[target].copy()

groups = df["Num_Acc"]

print("Variables explicatives :", X.shape[1])
print("Classes à prédire :", y.unique())

Variables explicatives : 15
Classes à prédire : <StringArray>
['Blessé léger', 'Indemne', 'Blessé hospitalisé', 'Tué']
Length: 4, dtype: str


In [8]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_groups = set(groups.iloc[train_idx])
test_groups = set(groups.iloc[test_idx])

print("Observations entraînement :", len(X_train))
print("Observations test :", len(X_test))

print("Accidents entraînement :", len(train_groups))
print("Accidents test :", len(test_groups))

print(
    "Accidents communs entre entraînement et test :",
    len(train_groups.intersection(test_groups))
)

Observations entraînement : 525992
Observations test : 131416
Accidents entraînement : 218580
Accidents test : 54646
Accidents communs entre entraînement et test : 0


In [9]:
distribution = pd.DataFrame({
    "Entraînement (%)": (
        y_train.value_counts(normalize=True) * 100
    ),
    "Test (%)": (
        y_test.value_counts(normalize=True) * 100
    )
}).round(2)

display(distribution)

,Entraînement (%),Test (%)
grav,,
Indemne,42.27,42.53
Blessé léger,39.94,39.82
Blessé hospitalisé,15.21,15.08
Tué,2.58,2.57


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

numeric_features = ["age", "nbv"]

categorical_features = [
    "sexe", "catu", "catv", "atm", "lum",
    "col", "choc", "manv", "plan", "surf",
    "secu1", "infra", "place"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Non renseigné"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Pipeline de prétraitement créé.")

Pipeline de prétraitement créé.


In [11]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ))
])

print("Random Forest prêt pour l'entraînement.")

Random Forest prêt pour l'entraînement.


In [12]:
import time

start_time = time.time()

rf_pipeline.fit(X_train, y_train)

training_time = time.time() - start_time

print(f"Entraînement terminé en {training_time:.1f} secondes.")

Entraînement terminé en 69.0 secondes.


In [13]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

labels = [
    "Indemne",
    "Blessé léger",
    "Blessé hospitalisé",
    "Tué"
]

y_pred = rf_pipeline.predict(X_test)

print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))

print("\nRapport de classification :")
print(
    classification_report(
        y_test,
        y_pred,
        labels=labels,
        zero_division=0
    )
)

print("\nMatrice de confusion :")

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels
)

display(
    pd.DataFrame(
        cm,
        index=[f"Réel : {label}" for label in labels],
        columns=[f"Prédit : {label}" for label in labels]
    )
)

Accuracy : 0.5685

Rapport de classification :
                    precision    recall  f1-score   support

           Indemne       0.69      0.80      0.74     55891
      Blessé léger       0.68      0.39      0.50     52331
Blessé hospitalisé       0.36      0.38      0.37     19821
               Tué       0.13      0.56      0.20      3373

          accuracy                           0.57    131416
         macro avg       0.46      0.53      0.45    131416
      weighted avg       0.62      0.57      0.57    131416


Matrice de confusion :


,Prédit : Indemne,Prédit : Blessé léger,Prédit : Blessé hospitalisé,Prédit : Tué
Réel : Indemne,44828,5566,2342,3155
Réel : Blessé léger,16432,20493,10509,4897
Réel : Blessé hospitalisé,3348,3926,7505,5042
Réel : Tué,500,251,744,1878


In [15]:
from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder

# LightGBM nécessite ici des étiquettes de classes numériques
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

lgbm_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

print("Classes :", label_encoder.classes_)
print("Pipeline LightGBM prêt.")

Classes : ['Blessé hospitalisé' 'Blessé léger' 'Indemne' 'Tué']
Pipeline LightGBM prêt.


In [16]:
import time

start_time = time.time()

lgbm_pipeline.fit(X_train, y_train_encoded)

training_time = time.time() - start_time

print(f"Entraînement LightGBM terminé en {training_time:.1f} secondes.")

Entraînement LightGBM terminé en 11.6 secondes.


In [17]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

y_pred_encoded = lgbm_pipeline.predict(X_test)
y_pred_lgbm = label_encoder.inverse_transform(y_pred_encoded)

print("Accuracy :", round(accuracy_score(y_test, y_pred_lgbm), 4))
print("F1 macro :", round(f1_score(y_test, y_pred_lgbm, average="macro"), 4))

print("\nRapport de classification :")
print(classification_report(
    y_test,
    y_pred_lgbm,
    labels=labels,
    zero_division=0
))

Accuracy : 0.5655
F1 macro : 0.4572

Rapport de classification :
                    precision    recall  f1-score   support

           Indemne       0.73      0.74      0.73     55891
      Blessé léger       0.66      0.44      0.53     52331
Blessé hospitalisé       0.35      0.38      0.37     19821
               Tué       0.12      0.62      0.20      3373

          accuracy                           0.57    131416
         macro avg       0.46      0.55      0.46    131416
      weighted avg       0.63      0.57      0.58    131416



In [18]:
from xgboost import XGBClassifier

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    ))
])

print("Pipeline XGBoost prêt.")

Pipeline XGBoost prêt.


In [19]:
import time

start_time = time.time()

xgb_pipeline.fit(X_train, y_train_encoded)

training_time_xgb = time.time() - start_time

print(f"Entraînement XGBoost terminé en {training_time_xgb:.1f} secondes.")

Entraînement XGBoost terminé en 21.2 secondes.


In [20]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

y_pred_xgb_encoded = xgb_pipeline.predict(X_test)
y_pred_xgb = label_encoder.inverse_transform(
    y_pred_xgb_encoded.astype(int)
)

print("Accuracy :", round(accuracy_score(y_test, y_pred_xgb), 4))
print("F1 macro :", round(f1_score(y_test, y_pred_xgb, average="macro"), 4))

print("\nRapport de classification :")
print(classification_report(
    y_test,
    y_pred_xgb,
    labels=labels,
    zero_division=0
))

cm_xgb = confusion_matrix(y_test, y_pred_xgb, labels=labels)

display(pd.DataFrame(
    cm_xgb,
    index=[f"Réel : {label}" for label in labels],
    columns=[f"Prédit : {label}" for label in labels]
))

Accuracy : 0.637
F1 macro : 0.4354

Rapport de classification :
                    precision    recall  f1-score   support

           Indemne       0.68      0.83      0.75     55891
      Blessé léger       0.61      0.63      0.62     52331
Blessé hospitalisé       0.49      0.21      0.30     19821
               Tué       0.46      0.04      0.08      3373

          accuracy                           0.64    131416
         macro avg       0.56      0.43      0.44    131416
      weighted avg       0.62      0.64      0.61    131416



,Prédit : Indemne,Prédit : Blessé léger,Prédit : Blessé hospitalisé,Prédit : Tué
Réel : Indemne,46342,8762,771,16
Réel : Blessé léger,16809,32994,2472,56
Réel : Blessé hospitalisé,4338,11154,4231,98
Réel : Tué,870,1237,1122,144


In [21]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_encoded
)

print("Poids calculés pour les observations d'entraînement.")

Poids calculés pour les observations d'entraînement.


In [22]:
from sklearn.base import clone
import time

xgb_weighted_pipeline = clone(xgb_pipeline)

start_time = time.time()

xgb_weighted_pipeline.fit(
    X_train,
    y_train_encoded,
    classifier__sample_weight=sample_weights
)

print(
    f"Entraînement XGBoost pondéré terminé "
    f"en {time.time() - start_time:.1f} secondes."
)

Entraînement XGBoost pondéré terminé en 22.4 secondes.


In [23]:
y_pred_weighted_encoded = xgb_weighted_pipeline.predict(X_test)

y_pred_weighted = label_encoder.inverse_transform(
    y_pred_weighted_encoded.astype(int)
)

print("Accuracy :", round(accuracy_score(y_test, y_pred_weighted), 4))
print("F1 macro :", round(f1_score(y_test, y_pred_weighted, average="macro"), 4))

print("\nRapport de classification :")
print(
    classification_report(
        y_test,
        y_pred_weighted,
        labels=labels,
        zero_division=0
    )
)

Accuracy : 0.5632
F1 macro : 0.4526

Rapport de classification :
                    precision    recall  f1-score   support

           Indemne       0.71      0.76      0.74     55891
      Blessé léger       0.67      0.42      0.51     52331
Blessé hospitalisé       0.35      0.38      0.36     19821
               Tué       0.12      0.60      0.20      3373

          accuracy                           0.56    131416
         macro avg       0.46      0.54      0.45    131416
      weighted avg       0.63      0.56      0.58    131416



In [24]:
from sklearn.model_selection import GroupShuffleSplit

# Première séparation : 80 % développement, 20 % test
split_1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=123
)

dev_idx, final_test_idx = next(
    split_1.split(X, y, groups=groups)
)

# Deuxième séparation : 75 % train, 25 % validation
# sur les 80 % de développement
split_2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=456
)

train_sub_idx, val_sub_idx = next(
    split_2.split(
        X.iloc[dev_idx],
        y.iloc[dev_idx],
        groups=groups.iloc[dev_idx]
    )
)

train_idx = dev_idx[train_sub_idx]
val_idx = dev_idx[val_sub_idx]

X_train = X.iloc[train_idx].copy()
X_val = X.iloc[val_idx].copy()
X_test = X.iloc[final_test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_val = y.iloc[val_idx].copy()
y_test = y.iloc[final_test_idx].copy()

# Vérification des accidents communs
train_accidents = set(groups.iloc[train_idx])
val_accidents = set(groups.iloc[val_idx])
test_accidents = set(groups.iloc[final_test_idx])

print("Train :", len(X_train))
print("Validation :", len(X_val))
print("Test :", len(X_test))

print("Train / Validation :", len(train_accidents & val_accidents))
print("Train / Test :", len(train_accidents & test_accidents))
print("Validation / Test :", len(val_accidents & test_accidents))

Train : 394368
Validation : 131564
Test : 131476
Train / Validation : 0
Train / Test : 0
Validation / Test : 0


In [25]:
from sklearn.base import clone
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import time

# Encoder les étiquettes sur le nouvel ensemble d'entraînement
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

# Nouveau modèle indépendant de l'ancien
lgbm_v2 = clone(lgbm_pipeline)

start_time = time.time()

lgbm_v2.fit(X_train, y_train_encoded)

print(
    f"Entraînement terminé en "
    f"{time.time() - start_time:.1f} secondes."
)

Entraînement terminé en 20.0 secondes.


In [26]:
# Prédictions sur la validation uniquement
y_val_pred_encoded = lgbm_v2.predict(X_val)

y_val_pred = label_encoder.inverse_transform(
    y_val_pred_encoded.astype(int)
)

# Calcul des performances
print(
    "Accuracy :",
    round(accuracy_score(y_val, y_val_pred), 4)
)

print(
    "F1 macro :",
    round(f1_score(y_val, y_val_pred, average="macro"), 4)
)

print("\nRapport de classification :")

labels = [
    "Indemne",
    "Blessé léger",
    "Blessé hospitalisé",
    "Tué"
]

print(
    classification_report(
        y_val,
        y_val_pred,
        labels=labels,
        zero_division=0
    )
)

# Matrice de confusion
cm = confusion_matrix(
    y_val,
    y_val_pred,
    labels=labels
)

display(
    pd.DataFrame(
        cm,
        index=[f"Réel : {label}" for label in labels],
        columns=[f"Prédit : {label}" for label in labels]
    )
)

Accuracy : 0.5661
F1 macro : 0.4584

Rapport de classification :
                    precision    recall  f1-score   support

           Indemne       0.73      0.74      0.73     55650
      Blessé léger       0.67      0.44      0.53     52535
Blessé hospitalisé       0.34      0.39      0.37     19849
               Tué       0.12      0.58      0.20      3530

          accuracy                           0.57    131564
         macro avg       0.46      0.54      0.46    131564
      weighted avg       0.63      0.57      0.58    131564



,Prédit : Indemne,Prédit : Blessé léger,Prédit : Blessé hospitalisé,Prédit : Tué
Réel : Indemne,41363,7138,3316,3833
Réel : Blessé léger,12966,23232,10727,5610
Réel : Blessé hospitalisé,2317,4112,7833,5587
Réel : Tué,329,313,837,2051


In [27]:
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
import pandas as pd

# Classes encodées par LabelEncoder
print(dict(enumerate(label_encoder.classes_)))

# Référence : modèle équilibré déjà entraîné
modeles = {
    "Sans pondération": None,
    "Pondération modérée": "moderate",
    "Pondération équilibrée": "balanced"
}

resultats = []
modeles_entraines = {}

for nom, ponderation in modeles.items():

    if ponderation == "balanced":
        modele = lgbm_v2

    else:
        modele = clone(lgbm_pipeline)

        if ponderation is None:
            poids = None
        else:
            # Pondération intermédiaire entre 1 et les poids équilibrés
            effectifs = pd.Series(y_train_encoded).value_counts()
            n_classes = len(effectifs)
            n_total = len(y_train_encoded)

            poids_balances = {
                int(classe): n_total / (n_classes * effectif)
                for classe, effectif in effectifs.items()
            }

            poids = {
                classe: valeur ** 0.5
                for classe, valeur in poids_balances.items()
            }

        modele.set_params(classifier__class_weight=poids)
        modele.fit(X_train, y_train_encoded)

    modeles_entraines[nom] = modele

    prediction_encodee = modele.predict(X_val)
    prediction = label_encoder.inverse_transform(
        prediction_encodee.astype(int)
    )

    resultats.append({
        "Modèle": nom,
        "Accuracy": accuracy_score(y_val, prediction),
        "F1 macro": f1_score(
            y_val, prediction, average="macro"
        ),
        "Précision Tué": precision_score(
            y_val == "Tué",
            prediction == "Tué",
            zero_division=0
        ),
        "Rappel Tué": recall_score(
            y_val == "Tué",
            prediction == "Tué",
            zero_division=0
        )
    })

comparaison_poids = pd.DataFrame(resultats)

display(
    comparaison_poids.round(4).sort_values(
        "F1 macro",
        ascending=False
    )
)

{0: 'Blessé hospitalisé', 1: 'Blessé léger', 2: 'Indemne', 3: 'Tué'}


,Modèle,Accuracy,F1 macro,Précision Tué,Rappel Tué
1,Pondération modérée,0.6286,0.4978,0.2548,0.2130
2,Pondération équilibrée,0.5661,0.4584,0.1201,0.5810
0,Sans pondération,0.6395,0.4473,0.4238,0.0465


In [28]:
from sklearn.metrics import classification_report, confusion_matrix

# Récupérer le modèle à pondération modérée
best_model = modeles_entraines["Pondération modérée"]

# Prédire uniquement sur la validation
y_pred_encoded = best_model.predict(X_val)

y_pred = label_encoder.inverse_transform(
    y_pred_encoded.astype(int)
)

labels = [
    "Indemne",
    "Blessé léger",
    "Blessé hospitalisé",
    "Tué"
]

print("Rapport de classification :")
print(
    classification_report(
        y_val,
        y_pred,
        labels=labels,
        zero_division=0
    )
)

print("\nMatrice de confusion :")

cm = confusion_matrix(
    y_val,
    y_pred,
    labels=labels
)

display(
    pd.DataFrame(
        cm,
        index=[f"Réel : {label}" for label in labels],
        columns=[f"Prédit : {label}" for label in labels]
    )
)

Rapport de classification :
                    precision    recall  f1-score   support

           Indemne       0.71      0.79      0.75     55650
      Blessé léger       0.64      0.57      0.60     52535
Blessé hospitalisé       0.41      0.41      0.41     19849
               Tué       0.25      0.21      0.23      3530

          accuracy                           0.63    131564
         macro avg       0.50      0.50      0.50    131564
      weighted avg       0.62      0.63      0.62    131564


Matrice de confusion :


,Prédit : Indemne,Prédit : Blessé léger,Prédit : Blessé hospitalisé,Prédit : Tué
Réel : Indemne,43983,8697,2552,418
Réel : Blessé léger,14447,29800,7589,699
Réel : Blessé hospitalisé,3155,7450,8162,1082
Réel : Tué,571,719,1488,752


In [29]:
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
import pandas as pd

# Calculer les poids équilibrés sur le train uniquement
effectifs = pd.Series(y_train_encoded).value_counts()

n_total = len(y_train_encoded)
n_classes = len(effectifs)

poids_balances = {
    int(classe): n_total / (n_classes * effectif)
    for classe, effectif in effectifs.items()
}

# Intensités de pondération à comparer
exposants = [0.25, 0.50, 0.65, 0.80, 1.00]

resultats_ponderations = []
modeles_ponderations = {}

for exposant in exposants:

    poids = {
        classe: valeur ** exposant
        for classe, valeur in poids_balances.items()
    }

    # Réutiliser le modèle modéré déjà entraîné
    if exposant == 0.50:
        modele = modeles_entraines["Pondération modérée"]

    # Réutiliser le modèle équilibré déjà entraîné
    elif exposant == 1.00:
        modele = lgbm_v2

    else:
        modele = clone(lgbm_pipeline)

        modele.set_params(
            classifier__class_weight=poids
        )

        modele.fit(X_train, y_train_encoded)

    modeles_ponderations[exposant] = modele

    # Prédiction sur la validation uniquement
    pred_encoded = modele.predict(X_val)

    pred = label_encoder.inverse_transform(
        pred_encoded.astype(int)
    )

    resultats_ponderations.append({
        "Exposant": exposant,
        "Accuracy": accuracy_score(y_val, pred),
        "F1 macro": f1_score(
            y_val, pred, average="macro"
        ),
        "Précision Tué": precision_score(
            y_val == "Tué",
            pred == "Tué",
            zero_division=0
        ),
        "Rappel Tué": recall_score(
            y_val == "Tué",
            pred == "Tué",
            zero_division=0
        )
    })

comparaison_ponderations = pd.DataFrame(
    resultats_ponderations
)

display(
    comparaison_ponderations
    .sort_values("F1 macro", ascending=False)
    .round(4)
)

,Exposant,Accuracy,F1 macro,Précision Tué,Rappel Tué
2,0.65,0.6180,0.4982,0.2070,0.3099
1,0.50,0.6286,0.4978,0.2548,0.2130
3,0.80,0.6012,0.4858,0.1618,0.4184
0,0.25,0.6376,0.4815,0.3601,0.1156
4,1.00,0.5661,0.4584,0.1201,0.5810


In [30]:
from sklearn.metrics import classification_report, confusion_matrix

# Modèle LightGBM avec exposant 0.65
best_model = modeles_ponderations[0.65]

# Prédictions sur la validation
y_pred_encoded = best_model.predict(X_val)

y_pred = label_encoder.inverse_transform(
    y_pred_encoded.astype(int)
)

labels = [
    "Indemne",
    "Blessé léger",
    "Blessé hospitalisé",
    "Tué"
]

print("Rapport de classification :")
print(
    classification_report(
        y_val,
        y_pred,
        labels=labels,
        zero_division=0
    )
)

print("\nMatrice de confusion :")

cm = confusion_matrix(
    y_val,
    y_pred,
    labels=labels
)

display(
    pd.DataFrame(
        cm,
        index=[f"Réel : {label}" for label in labels],
        columns=[f"Prédit : {label}" for label in labels]
    )
)

Rapport de classification :
                    precision    recall  f1-score   support

           Indemne       0.71      0.78      0.75     55650
      Blessé léger       0.65      0.54      0.59     52535
Blessé hospitalisé       0.39      0.43      0.41     19849
               Tué       0.21      0.31      0.25      3530

          accuracy                           0.62    131564
         macro avg       0.49      0.52      0.50    131564
      weighted avg       0.63      0.62      0.62    131564


Matrice de confusion :


,Prédit : Indemne,Prédit : Blessé léger,Prédit : Blessé hospitalisé,Prédit : Tué
Réel : Indemne,43439,8325,3003,883
Réel : Blessé léger,14074,28161,8902,1398
Réel : Blessé hospitalisé,2916,6407,8615,1911
Réel : Tué,492,578,1366,1094


In [31]:
from sklearn.base import clone
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import pandas as pd
import time

# Regrouper train et validation
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

# Encoder les classes avec le même LabelEncoder
y_train_final_encoded = label_encoder.transform(y_train_final)

# Calculer les poids sur train + validation
classes = np.unique(y_train_final_encoded)

poids_balances = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_final_encoded
)

poids_moderes = {
    int(classe): float(poids ** 0.65)
    for classe, poids in zip(classes, poids_balances)
}

print("Poids retenus :", poids_moderes)

# Créer un nouveau modèle
final_model = clone(lgbm_pipeline)

final_model.set_params(
    classifier__class_weight=poids_moderes
)

# Entraînement final
start_time = time.time()

final_model.fit(
    X_train_final,
    y_train_final_encoded
)

print(
    f"Entraînement final terminé en "
    f"{time.time() - start_time:.1f} secondes."
)

Poids retenus : {0: 1.3845551212856146, 1: 0.7372471071376404, 2: 0.7104088212958438, 3: 4.374016254084539}
Entraînement final terminé en 14.0 secondes.


In [32]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Prédiction sur le test final
y_test_pred_encoded = final_model.predict(X_test)

y_test_pred = label_encoder.inverse_transform(
    y_test_pred_encoded.astype(int)
)

labels = [
    "Indemne",
    "Blessé léger",
    "Blessé hospitalisé",
    "Tué"
]

print("Accuracy :", round(
    accuracy_score(y_test, y_test_pred), 4
))

print("F1 macro :", round(
    f1_score(y_test, y_test_pred, average="macro"), 4
))

print("\nRapport de classification :")
print(classification_report(
    y_test,
    y_test_pred,
    labels=labels,
    zero_division=0
))

cm_final = confusion_matrix(
    y_test,
    y_test_pred,
    labels=labels
)

display(pd.DataFrame(
    cm_final,
    index=[f"Réel : {label}" for label in labels],
    columns=[f"Prédit : {label}" for label in labels]
))

Accuracy : 0.6186
F1 macro : 0.4979

Rapport de classification :
                    precision    recall  f1-score   support

           Indemne       0.71      0.78      0.75     55722
      Blessé léger       0.65      0.54      0.59     52265
Blessé hospitalisé       0.40      0.43      0.42     20141
               Tué       0.20      0.32      0.24      3348

          accuracy                           0.62    131476
         macro avg       0.49      0.52      0.50    131476
      weighted avg       0.63      0.62      0.62    131476



,Prédit : Indemne,Prédit : Blessé léger,Prédit : Blessé hospitalisé,Prédit : Tué
Réel : Indemne,43606,8220,2968,928
Réel : Blessé léger,14120,27990,8724,1431
Réel : Blessé hospitalisé,3107,6452,8680,1902
Réel : Tué,444,608,1236,1060


In [33]:
import joblib
from pathlib import Path

# Dossier de sauvegarde
MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Sauvegarde du pipeline complet
joblib.dump(
    final_model,
    MODELS_DIR / "lightgbm_gravite.joblib"
)

# Sauvegarde de l'encodeur des gravités
joblib.dump(
    label_encoder,
    MODELS_DIR / "gravite_encoder.joblib"
)

print("Modèle et encodeur sauvegardés dans :", MODELS_DIR)

Modèle et encodeur sauvegardés dans : c:\Users\david\Documents\David\Etudes\Projet\M2\OpenData\car_accident_od\models


In [55]:
# Recharger le modèle et l'encodeur
loaded_model = joblib.load(
    MODELS_DIR / "lightgbm_gravite.joblib"
)

loaded_encoder = joblib.load(
    MODELS_DIR / "gravite_encoder.joblib"
)

# Prendre une observation de validation comme exemple
exemple = X_val.iloc[[0]]

# Prédiction
prediction_encoded = loaded_model.predict(exemple)

prediction = loaded_encoder.inverse_transform(
    prediction_encoded.astype(int)
)

print("Prédiction :", prediction[0])
print("Gravité réelle :", y_val.iloc[0])

Prédiction : Blessé léger
Gravité réelle : Blessé léger
